# Naive Bayes for Conspiracy Detection
## Binary classification (yes/no only)

This notebook implements Naive Bayes classifier for:
1. **Conspiracy Label Prediction**: **binary (yes/no only)** — "cant_tell" samples are excluded (same as train_and_infer_binary.py)
2. **Model Evaluation**: 4-fold cross-validation on training set
3. **Validation Set Testing**: Evaluation on held-out validation set (100 samples)


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import json
import warnings
warnings.filterwarnings('ignore')

from sklearn.naive_bayes import MultinomialNB, GaussianNB
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_validate
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("Libraries imported successfully!")


In [ ]:
# Load and merge all features
BASE = Path('../')
PROC = BASE / 'data_processed'
FEAT = BASE / 'features'

# Load base data from data_clean.csv (contains _id, text, conspiracy)
df = pd.read_csv(PROC / 'data_clean.csv')

# Binary classification only: ignore "cant_tell" (same as train_and_infer_binary.py)
n_before = len(df)
df = df[df['conspiracy'].isin(['yes', 'no'])].copy()
n_after = len(df)
print(f"Binary classification: kept only yes/no, excluded {n_before - n_after} 'cant_tell' samples")
print(f"Base data: {df.shape}")
print(f"Base columns: {list(df.columns)}")
print(f"Base data row count: {len(df)}")

# Preserve original columns from data_clean.csv
base_cols = ['_id', 'text', 'conspiracy']
for col in base_cols:
    if col not in df.columns:
        print(f"WARNING: {col} not found in base data!")

# Store base _id values to ensure we only keep these rows
base_ids = set(df['_id'].values)
print(f"Unique _id values in base: {len(base_ids)}")

# Load feature files
feature_files = [
    FEAT / 'lexical_complexity.csv',
    FEAT / 'discourse_markers.csv',
    FEAT / 'sentiment_emotion.csv',
    FEAT / 'token_pos_counts.csv',
    FEAT / 'readability_scores.csv',
    FEAT / 'ma_ttr_mtld_scores.csv',
    FEAT / 'pos_analysis.csv'
]

# Start with base data - this preserves _id, text, conspiracy from data_clean.csv
merged = df.copy()
initial_row_count = len(merged)

# Merge feature files, ensuring we keep original columns from data_clean.csv
# and preserve the exact row count from base data
for feat_file in feature_files:
    if feat_file.exists():
        try:
            feat_df = pd.read_csv(feat_file)
            print(f"\nLoading {feat_file.name}: {feat_df.shape[0]} rows, {feat_df.shape[1]} columns")
            
            if '_id' in merged.columns and '_id' in feat_df.columns:
                # Filter feature file to only include rows with _id in base data
                # This ensures we only add columns horizontally, preserving base row count
                feat_df_filtered = feat_df[feat_df['_id'].isin(base_ids)].copy()
                print(f"  Filtered to {len(feat_df_filtered)} rows matching base _id values")
                
                # Remove duplicates by _id (keep first occurrence) to prevent row explosion
                # This handles cases like readability_scores.csv which has duplicate _id values
                initial_filtered_count = len(feat_df_filtered)
                feat_df_filtered = feat_df_filtered.drop_duplicates(subset=['_id'], keep='first')
                if len(feat_df_filtered) < initial_filtered_count:
                    print(f"  Removed {initial_filtered_count - len(feat_df_filtered)} duplicate _id rows (kept first)")
                
                # Drop any columns from feature file that conflict with base_cols (except _id)
                # This ensures text and conspiracy from data_clean.csv are preserved
                cols_to_drop = [col for col in feat_df_filtered.columns if col in base_cols and col != '_id']
                if cols_to_drop:
                    feat_df_filtered = feat_df_filtered.drop(columns=cols_to_drop)
                    print(f"  Dropped conflicting columns: {cols_to_drop}")
                
                # Merge on _id, keeping all rows from merged (left join)
                # This ensures row count stays the same as base data
                merged = merged.merge(feat_df_filtered, on='_id', how='left')
                print(f"  Merged successfully. Merged shape: {merged.shape}")
            else:
                print(f"  Skipped: missing '_id' column")
        except Exception as e:
            print(f"Could not load {feat_file.name}: {e}")

# Verify that original columns from data_clean.csv are still present
for col in base_cols:
    if col not in merged.columns:
        print(f"ERROR: {col} was lost during merge!")

# Verify row count is preserved
if len(merged) != initial_row_count:
    print(f"\nWARNING: Row count changed from {initial_row_count} to {len(merged)}!")
else:
    print(f"\n✓ Row count preserved: {initial_row_count}")

print(f"\nFinal merged shape: {merged.shape}")
print(f"Total columns: {len(merged.columns)}")
print(f"Original columns preserved: {all(col in merged.columns for col in base_cols)}")
print(f"\nColumn breakdown:")
print(f"  Base columns: {base_cols}")
print(f"  Feature columns added: {len(merged.columns) - len(base_cols)}")
print(f"\nFirst few rows (showing base columns + first 5 feature columns):")
feature_cols = [col for col in merged.columns if col not in base_cols][:5]
print(merged[base_cols + feature_cols].head())


In [ ]:
# Create validation set (100 randomly selected points)
# This validation set will be saved and reused across all ML notebooks
from sklearn.model_selection import train_test_split

VALIDATION_SIZE = 100
VALIDATION_FILE = PROC / 'validation_set_ids.csv'

# Set random seed for reproducibility
np.random.seed(42)

# Check if validation set already exists
if VALIDATION_FILE.exists():
    print("Loading existing validation set...")
    validation_ids_df = pd.read_csv(VALIDATION_FILE)
    validation_ids = set(validation_ids_df['_id'].values)
    print(f"  Loaded {len(validation_ids)} validation IDs from {VALIDATION_FILE.name}")
else:
    print(f"Creating new validation set ({VALIDATION_SIZE} samples)...")
    
    # Use stratified sampling to maintain class distribution
    # First, check if we have enough samples per class
    label_counts = merged['conspiracy'].value_counts()
    print(f"  Class distribution: {dict(label_counts)}")
    
    # Calculate stratified split - we want exactly 100 samples
    # Using a fraction to get approximately 100, then adjusting
    target_fraction = VALIDATION_SIZE / len(merged)
    
    # Perform stratified split
    train_ids, val_ids = train_test_split(
        merged['_id'].values,
        test_size=target_fraction,
        stratify=merged['conspiracy'].values,
        random_state=42
    )
    
    # Adjust to exactly 100 if needed (if we got more or less)
    if len(val_ids) > VALIDATION_SIZE:
        # Randomly select exactly 100 from validation set, maintaining stratification
        val_df_temp = merged[merged['_id'].isin(val_ids)]
        val_ids_adjusted, _ = train_test_split(
            val_ids,
            test_size=VALIDATION_SIZE / len(val_ids),
            stratify=val_df_temp['conspiracy'].values,
            random_state=42
        )
        validation_ids = set(val_ids_adjusted)
    elif len(val_ids) < VALIDATION_SIZE:
        # Sample more from training set, maintaining stratification
        train_df_temp = merged[merged['_id'].isin(train_ids)]
        additional_needed = VALIDATION_SIZE - len(val_ids)
        additional_ids, _ = train_test_split(
            train_ids,
            test_size=additional_needed / len(train_ids),
            stratify=train_df_temp['conspiracy'].values,
            random_state=42
        )
        validation_ids = set(list(val_ids) + list(additional_ids))
    else:
        validation_ids = set(val_ids)
    
    # Save validation set IDs
    validation_ids_df = pd.DataFrame({'_id': list(validation_ids)})
    validation_ids_df.to_csv(VALIDATION_FILE, index=False)
    print(f"  ✓ Saved validation set to {VALIDATION_FILE.name}")
    print(f"  Validation set size: {len(validation_ids)} samples")

# Display validation set distribution
validation_df = merged[merged['_id'].isin(validation_ids)]
print(f"\nValidation set class distribution:")
print(validation_df['conspiracy'].value_counts().to_dict())

# Remove validation set from training data
train_df = merged[~merged['_id'].isin(validation_ids)].copy()
print(f"\nTraining set size: {len(train_df)} samples")
print(f"Training set class distribution:")
print(train_df['conspiracy'].value_counts().to_dict())

# Update merged to be training data only
merged = train_df.copy()
print(f"\n✓ Updated merged dataframe for training: {merged.shape}")


In [ ]:
# Extract numeric features
numeric_cols = merged.select_dtypes(include=[np.number]).columns.tolist()
# Remove _id and other non-feature columns
numeric_cols = [c for c in numeric_cols if c not in ['_id']]
print(f"Number of numeric features: {len(numeric_cols)}")

X = merged[numeric_cols].fillna(0)
print(X.head())
print(f"Feature matrix shape: {X.shape}")
print(f"Missing values: {X.isna().sum().sum()}")


## Task 1: Conspiracy Label Prediction (binary: yes/no only)


In [ ]:
# Task 1: Conspiracy Label Prediction (binary: yes/no only)
y_conspiracy = merged['conspiracy'].copy()
# Filter out any null labels (data is already restricted to yes/no in cell 2)
mask = y_conspiracy.notna()
X_clean = X[mask]
y_clean = y_conspiracy[mask]

print(f"Samples for conspiracy prediction (binary): {len(y_clean)}")
print(f"Label distribution (yes/no only):")
print(y_clean.value_counts())


In [ ]:
# Encode labels
le_conspiracy = LabelEncoder()
y_encoded = le_conspiracy.fit_transform(y_clean)
print(f"Encoded labels: {dict(zip(le_conspiracy.classes_, range(len(le_conspiracy.classes_))))}")

# Create pipeline with scaling and Naive Bayes
# Use GaussianNB for continuous features
pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler()),
    ('nb', GaussianNB())
])

# 4-fold cross-validation
cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=42)

scoring = ['accuracy', 'f1_macro', 'f1_weighted', 'precision_macro', 'recall_macro']
cv_results = cross_validate(pipeline, X_clean, y_encoded, cv=cv, scoring=scoring, n_jobs=-1)

print("\n=== Conspiracy Label Prediction - Binary yes/no (4-fold CV) ===")
for metric in scoring:
    scores = cv_results[f'test_{metric}']
    print(f"{metric}: {scores.mean():.4f} (+/- {scores.std() * 2:.4f})")

# Train final model
pipeline.fit(X_clean, y_encoded)
print("\nModel trained successfully!")


## Testing on Validation Set

Evaluate the trained model on the held-out validation set (100 samples).


In [ ]:
# Load validation set
validation_ids_df = pd.read_csv(VALIDATION_FILE)
validation_ids = set(validation_ids_df['_id'].values)

# Get validation set data from the original merged dataframe (before splitting)
# We need to reload the full dataset to get validation samples
df_full = pd.read_csv(PROC / 'data_clean.csv')

# Merge features for validation set
validation_df_base = df_full[df_full['_id'].isin(validation_ids)].copy()
print(f"Validation set base size: {len(validation_df_base)}")

# Merge all features for validation set (same as training data)
validation_merged = validation_df_base.copy()

# Load and merge feature files for validation set
for feat_file in feature_files:
    if feat_file.exists():
        try:
            feat_df = pd.read_csv(feat_file)
            if '_id' in validation_merged.columns and '_id' in feat_df.columns:
                feat_df_filtered = feat_df[feat_df['_id'].isin(validation_ids)].copy()
                # Remove duplicates
                feat_df_filtered = feat_df_filtered.drop_duplicates(subset=['_id'], keep='first')
                # Drop conflicting columns
                cols_to_drop = [col for col in feat_df_filtered.columns if col in base_cols and col != '_id']
                if cols_to_drop:
                    feat_df_filtered = feat_df_filtered.drop(columns=cols_to_drop)
                validation_merged = validation_merged.merge(feat_df_filtered, on='_id', how='left')
        except Exception as e:
            print(f"Could not load {feat_file.name} for validation: {e}")

print(f"Validation set with features: {validation_merged.shape}")
print(f"Validation set class distribution (before binary filter):")
print(validation_merged['conspiracy'].value_counts())

# Binary only: keep only yes/no for evaluation (match train_and_infer_binary.py)
validation_merged = validation_merged[validation_merged['conspiracy'].isin(['yes', 'no'])].copy()
print(f"\nAfter excluding 'cant_tell': {len(validation_merged)} validation samples (binary only)")

# Prepare validation features
X_val = validation_merged[numeric_cols].fillna(0)
y_val = validation_merged['conspiracy'].copy()

# Encode validation labels using the same encoder (yes/no only)
y_val_encoded = le_conspiracy.transform(y_val)

print(f"\nValidation set feature matrix: {X_val.shape}")
print(f"Validation set labels: {len(y_val)}")


In [ ]:
# Make predictions on validation set
y_val_pred = pipeline.predict(X_val)
y_val_pred_labels = le_conspiracy.inverse_transform(y_val_pred)

print("=== Validation Set Predictions ===")
print(f"\nTrue label distribution:")
print(y_val.value_counts().to_dict())

print(f"\nPredicted label distribution:")
unique, counts = np.unique(y_val_pred_labels, return_counts=True)
print(dict(zip(unique, counts)))

# Calculate metrics
val_accuracy = accuracy_score(y_val_encoded, y_val_pred)
val_f1_macro = f1_score(y_val_encoded, y_val_pred, average='macro')
val_f1_weighted = f1_score(y_val_encoded, y_val_pred, average='weighted')

print(f"\n=== Validation Set Performance ===")
print(f"Accuracy: {val_accuracy:.4f}")
print(f"F1 (macro): {val_f1_macro:.4f}")
print(f"F1 (weighted): {val_f1_weighted:.4f}")

# Detailed classification report
print(f"\n=== Detailed Classification Report ===")
print(classification_report(y_val, y_val_pred_labels, target_names=le_conspiracy.classes_))

# Confusion matrix
cm = confusion_matrix(y_val_encoded, y_val_pred)
print(f"\n=== Confusion Matrix ===")
print("Rows = True labels, Columns = Predicted labels")
print(f"Label order: {list(le_conspiracy.classes_)}")
print(cm)

# Visualize confusion matrix
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=le_conspiracy.classes_, 
            yticklabels=le_conspiracy.classes_,
            cbar_kws={'label': 'Count'})
plt.title('Confusion Matrix - Validation Set', fontsize=14, fontweight='bold')
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.tight_layout()
plt.savefig(BASE / 'figures' / 'validation_confusion_matrix_nb.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"\n✓ Saved confusion matrix to figures/validation_confusion_matrix_nb.png")


In [ ]:
# Compare predictions with true labels for some examples
print("\n=== Sample Predictions ===")
sample_indices = np.random.choice(len(validation_merged), size=min(10, len(validation_merged)), replace=False)

results_df = pd.DataFrame({
    '_id': validation_merged.iloc[sample_indices]['_id'].values,
    'true_label': y_val.iloc[sample_indices].values,
    'predicted_label': y_val_pred_labels[sample_indices],
    'correct': (y_val.iloc[sample_indices].values == y_val_pred_labels[sample_indices])
})

print(results_df.to_string(index=False))

print(f"\n✓ Validation set evaluation complete!")
print(f"  Total validation samples: {len(validation_merged)}")
print(f"  Accuracy: {val_accuracy:.4f}")
print(f"  Correct predictions: {(y_val.values == y_val_pred_labels).sum()}/{len(y_val)}")
